# slm-rag -- Phi-4-mini RAG fine-tune (Google Colab)

LoRA fine-tunes **Phi-4-mini-instruct** on the slm-rag correction dataset with
[Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit **Q4_K_M GGUF**
(~2.5 GB), and uploads it to Hugging Face -- where `serve.py` auto-downloads it
on restart (P11 staleness check).

**Dataset shape.** Each training row comes from the slm-rag correction loop
(`training/rag_finetune.csv`, columns `question, context, answer`): a user
question, the exact context chunks that were retrieved, and the corrected
grounded answer.  The fine-tune reinforces two behaviours the base model
sometimes violates:
- answer **only** from the supplied context, never from parametric memory;
- say **"I don't know"** when the answer is not in the context.

Every example is rendered with the **same system prompt slm-rag uses at
inference**, so the model learns the exact behaviour the server expects.

### Notes that make Phi-4-mini work
- **6 epochs** -- Phi-4-mini needs more passes; fewer risks format inconsistency.
- **GGUF conversion workaround** -- Phi-4-mini uses a BPE/tiktoken tokenizer so
  Unsloth's bundled converter fails.  We merge to 16-bit, set
  `tokenizer_class="GPT2Tokenizer"`, convert with upstream llama.cpp, then
  quantize.

### Before you run
- **Runtime -> Change runtime type -> GPU** (a free T4 is enough).
- Add a Colab **secret** `HF_TOKEN` (key icon, left sidebar) with a Hugging
  Face **write** token.
- Place `training/rag_finetune.csv` in your repo (or use the inline sample
  below).  To smoke-test the whole notebook without a real dataset, set
  `USE_SAMPLE_CSV = True` in the *Dataset* cell.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
# Unsloth + a dependency set it supports.
# peft>=0.19.0 for newer arch LoRA targets. Drop the runtime's broken torchao.
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers>4.56.2,<=5.5.0,!=5.0.0,!=5.1.0" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft>=0.19.0"
!pip uninstall -q -y torchao
import importlib.metadata as m
print("transformers", m.version("transformers"), "| trl", m.version("trl"),
      "| datasets", m.version("datasets"), "| peft", m.version("peft"))

In [ ]:
import os
# Eager guard: some Colab torch images ship a broken torch.compile that crashes
# Unsloth at import (ImportError: CUSTOM_KEY). Run eager to be safe.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048  # RAG prompts are longer than chat-only (context chunks included)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Phi-4-mini-instruct",
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded Phi-4-mini-instruct")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset -- grounded RAG prompt format

Each example is formatted with a **system prompt** that instructs Phi to:
1. Answer **only** from the supplied context.
2. Cite each claim with a letter chunk reference: `[A]` or `[B][C]`.
3. Reply with **"I don't know based on the provided documents"** when the answer
   is absent from the context.

This matches the prompt slm-rag sends at inference (see `serve.py`, P6).  Training
and serving use the **same template** so the model reinforces exactly the behaviour
the server expects -- grounded, cited answers and honest "I don't know" responses.

**Design note (Colab model x prompt eval, 2025-06).** The full grounding instruction
lives *only* in the system message; the user turn carries labelled chunks + a bare
`Question:` line with no repeated footer.  The eval showed that a redundant footer
caused over-refusal on answers stated in narrative/dialogue; moving it exclusively
to the system message fixes this while preserving correct refusal on out-of-corpus
questions.

**Context format.** Each chunk in the `context` column is formatted as serve.py
produces at inference:
```
[A] (source: <path>, chunk <idx>)
<chunk text>
```
Answers cite chunks by letter: `[A]` or `[B][C]`.

**Smoke-test flag.** Set `USE_SAMPLE_CSV = True` to train on the tiny inline
sample and verify the whole pipeline end-to-end on Colab without a real dataset.
Set it to `False` (the default) to load `training/rag_finetune.csv` from GitHub.

In [ ]:
import csv, io, textwrap
import urllib.request
from datasets import Dataset

# -----------------------------------------------------------------------
# Smoke-test flag -- set True to train on the inline sample CSV below
# (a few rows, runs end-to-end on Colab without a real dataset).
# Set False to load the real corrections export from GitHub.
# -----------------------------------------------------------------------
USE_SAMPLE_CSV = True  # <-- flip to False for a real fine-tune

# Context column uses the letter-labelled-chunk format that serve.py produces:
#   [A] (source: <path>, chunk <idx>)
#   <text>
# Answers cite chunks by letter: [A] or [B][C] (no [Source: ...] format).
SAMPLE_CSV = textwrap.dedent("""\
question,context,answer
What is the maximum upload size?,"[A] (source: faq.md, chunk 1)
The system accepts files up to 50 MB per upload. Larger files must be split before uploading.",The maximum upload size is 50 MB per file. [A]
How do I reset my password?,"[A] (source: user_guide.md, chunk 3)
To reset your password click Account > Security > Reset Password. You will receive an email within five minutes.",To reset your password go to Account > Security > Reset Password; you will receive a reset email within five minutes. [A]
What is the refund policy for annual plans?,"[A] (source: terms.md, chunk 7)
Monthly plans may be cancelled at any time for a prorated refund. Annual plans are non-refundable after the 14-day trial window.",Annual plans are non-refundable once the 14-day trial has ended. [A]
Does the product support Linux?,"[A] (source: install_guide.md, chunk 2)
Supported operating systems: Windows 10+, macOS 12+, Ubuntu 20.04+, Debian 11+.",Yes, Linux is supported. Ubuntu 20.04+ and Debian 11+ are the tested distributions. [A]
What is the depreciation schedule for equipment?,"[A] (source: faq.md, chunk 1)
The system accepts files up to 50 MB per upload.",I don't know based on the provided documents.
""")

# -----------------------------------------------------------------------
# System prompt -- MUST match the prompt serve.py uses at inference (P6).
# Fine-tuning and serving share the same template so training reinforces
# exactly the grounded, cited behaviour the server expects.
#
# DESIGN (chosen from Colab model x prompt eval, 2025-06):
#   The full grounding instruction lives ONLY in SYSTEM_PROMPT.  The user
#   turn carries numbered chunks + bare "Question: ..." with NO repeated
#   footer.  Moving the instruction exclusively to the system message fixes
#   over-refusal on answers stated in narrative/dialogue while preserving
#   correct "I don't know" behaviour on out-of-corpus questions.
# -----------------------------------------------------------------------
SYSTEM_PROMPT = (
    "You answer using ONLY the labelled chunks below. "
    "The answer may be stated indirectly, in narration or dialogue -- extract it and state it plainly "
    "in 1-3 sentences in your own words. "
    "Cite the chunk label(s) you used like [A] or [B][C]. "
    "Only if none of the chunks are relevant, reply with exactly: "
    "\"I don't know based on the provided documents.\""
)

def build_user_message(question, context):
    """Reproduce the exact user turn slm-rag sends to Phi at inference.

    context is the numbered-chunk block (pre-formatted as serve.py produces).
    No instruction footer -- the instruction lives solely in SYSTEM_PROMPT.
    """
    return (
        f"{context}\n\n"
        f"Question: {question}"
    )

def to_text(row):
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": build_user_message(row["question"], row["context"])},
        {"role": "assistant", "content": row["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False)}

if USE_SAMPLE_CSV:
    print("[SMOKE TEST] using inline sample CSV")
    rows = list(csv.DictReader(io.StringIO(SAMPLE_CSV)))
else:
    # Load the real corrections CSV from the slm-rag GitHub repo.
    # Adjust the URL if you host the CSV elsewhere.
    CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/slm-rag/main/training/rag_finetune.csv"
    raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(raw)))

print(f"Loaded {len(rows)} rows")
ds = Dataset.from_list([to_text(r) for r in rows])
print("Sample (first 600 chars):")
print(ds[0]["text"][:600])

## Training

6 epochs -- Phi-4-mini needs more passes than other models; fewer risks
inconsistent output format.  Batch 4 x gradient accumulation 4 = effective
batch 16.  Cosine schedule with 10 % warmup.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 6,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 10,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Sanity check -- grounded answering and "I don't know"

Verifies the fine-tuned model:
1. Answers **only from the context** when the answer is present (hard gate).
2. Returns the expected "I don't know" string when the answer is absent (hard gate).

If either gate fails, `Run all` stops here -- do **not** upload a model that
ignores its context.

In [ ]:
import re
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def ask_rag(question, context):
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": build_user_message(question, context)},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out    = model.generate(**inputs, max_new_tokens=300, temperature=0.1, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

IDK_PHRASE = "i don't know based on the provided documents"

# Gate 1: in-context questions -- answer must mention the expected key phrase
# and include a [L] letter citation marker.
# Context uses the letter-labelled-chunk format that serve.py produces.
IN_CTX = [
    (
        "What is the maximum upload size?",
        "[A] (source: faq.md, chunk 1)\nThe system accepts files up to 50 MB per upload.",
        "50",  # expected substring in answer
    ),
    (
        "How do I reset my password?",
        "[A] (source: user_guide.md, chunk 3)\nTo reset your password click Account > Security > Reset Password.",
        "Security",
    ),
]

# Gate 2: out-of-context question -- must say "I don't know"
OUT_CTX = [
    (
        "What is the capital of France?",
        "[A] (source: readme.md, chunk 1)\nThis document covers upload limits.",
    ),
    (
        "Who won the 2022 World Cup?",
        "[A] (source: terms.md, chunk 2)\nAnnual plans are non-refundable after 14 days.",
    ),
]

in_pass = out_pass = 0

print("=== Gate 1: in-context answers ===")
for q, ctx, expected in IN_CTX:
    ans = ask_rag(q, ctx)
    has_citation = bool(re.search(r"\[[A-Z]+\]", ans))
    has_content  = expected.lower() in ans.lower()
    ok = has_content and has_citation
    in_pass += ok
    label = "OK " if ok else "BAD"
    print(f"{label} | Q: {q[:50]} | citation={has_citation} content={has_content}")
    print(f"     Answer: {ans[:120]}")

print("\n=== Gate 2: out-of-context (must say I don't know) ===")
for q, ctx in OUT_CTX:
    ans = ask_rag(q, ctx)
    idk = IDK_PHRASE in ans.lower()
    out_pass += idk
    label = "OK " if idk else "BAD"
    print(f"{label} | Q: {q[:50]}")
    print(f"     Answer: {ans[:120]}")

print(f"\nGate 1 (in-context):  {in_pass}/{len(IN_CTX)}")
print(f"Gate 2 (out-of-ctx):  {out_pass}/{len(OUT_CTX)}")

assert in_pass == len(IN_CTX), \
    f"IN-CONTEXT REGRESSION: {in_pass}/{len(IN_CTX)} answers grounded+cited. Do NOT upload."
assert out_pass >= len(OUT_CTX) - 1, \
    f"IDK REGRESSION: only {out_pass}/{len(OUT_CTX)} out-of-context replies said I don't know. Do NOT upload."

print("\nGATE PASSED -- safe to export and upload.")

## Export Q4_K_M GGUF (manual -- Phi-4-mini tokenizer workaround)

Unsloth's bundled converter cannot handle Phi-4-mini's BPE/tiktoken tokenizer
(`Missing tokenizer.model`).  Workaround:
1. Merge LoRA adapters into 16-bit weights.
2. Set `tokenizer_class="GPT2Tokenizer"` so llama.cpp follows the BPE vocab path.
3. Convert to BF16 GGUF with upstream llama.cpp.
4. Quantize to Q4_K_M with Unsloth's prebuilt `llama-quantize`.

In [ ]:
import json, subprocess, os

MERGED_DIR  = "phi-rag-merged"
BF16_GGUF   = "/content/phi-rag-bf16.gguf"
Q4_GGUF     = "/content/phi-rag-q4_k_m.gguf"

# 1. Merge LoRA -> 16-bit
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# 2. Route llama.cpp to its BPE vocab path
cfg_path = f"{MERGED_DIR}/tokenizer_config.json"
cfg = json.load(open(cfg_path))
cfg["tokenizer_class"] = "GPT2Tokenizer"
json.dump(cfg, open(cfg_path, "w"), ensure_ascii=False, indent=2)

# 3. Upstream llama.cpp converter (bundled one rejects this tokenizer)
if not os.path.isdir("/content/llamacpp"):
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/ggml-org/llama.cpp",
                    "/content/llamacpp"], check=True)
    subprocess.run(["pip", "install", "-q", "gguf"], check=True)

r = subprocess.run(
    ["python", "/content/llamacpp/convert_hf_to_gguf.py",
     MERGED_DIR, "--outfile", BF16_GGUF, "--outtype", "bf16"],
    capture_output=True, text=True)
print("convert rc:", r.returncode, "|",
      (r.stderr.splitlines()[-1] if r.stderr else ""))
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError("GGUF conversion failed")

# 4. Quantize with Unsloth's prebuilt llama-quantize
subprocess.run(
    ["/root/.unsloth/llama.cpp/llama-quantize",
     BF16_GGUF, Q4_GGUF, "Q4_K_M"], check=True)

print("Q4_K_M GGUF:", round(os.path.getsize(Q4_GGUF)/1e9, 2), "GB")

## Upload to Hugging Face

Uploads the GGUF to `freeideas/phi4mini-rag` (your Hugging Face account).
The repo is created if it does not exist.  `serve.py`'s staleness check (P11)
compares the remote file size and SHA-256 sidecar on restart and pulls the new
weights automatically -- no manual copy needed.

**Requires the `HF_TOKEN` Colab secret** (key icon -> Secrets -> `HF_TOKEN`).

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_REPO = "freeideas/phi4mini-rag"   # your Hugging Face account / repo for the fine-tuned model
GGUF_FILENAME = "model-q4_k_m.gguf"  # filename serve.py looks for on HF

tok = userdata.get("HF_TOKEN")
api = HfApi()
api.create_repo(HF_REPO, repo_type="model", exist_ok=True, token=tok)
api.upload_file(
    path_or_fileobj = Q4_GGUF,
    path_in_repo    = GGUF_FILENAME,
    repo_id         = HF_REPO,
    repo_type       = "model",
    token           = tok,
)
print(f"Uploaded -> https://huggingface.co/{HF_REPO}")
print()
print("Next step: restart serve.py.")
print("  The P11 staleness check detects the updated weights on HF and downloads")
print("  them before launching llama-server, so the new fine-tuned model is live")
print("  automatically -- no other steps needed.")